In [0]:
# ===============================================================
# 02_Silver_Transformation_V2
# Camada Silver
# Databricks Serverless + Unity Catalog
# ===============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType,
    DateType,
    TimestampType
)

import time

# ===============================================================
# CONFIGURAÇÃO
# ===============================================================

BASE_PATH = "/Volumes/workspace/default/sources/delta"

PIPELINE_START = time.time()

dq_results = []

# ===============================================================
# LOAD
# ===============================================================

def load(table):

    path = f"{BASE_PATH}/{table}"

    print(f"Lendo {path}")

    return spark.read.format("delta").load(path)

# ===============================================================
# SAVE
# ===============================================================

def save(df, table):

    path = f"{BASE_PATH}/{table}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema","true")
        .save(path)
    )

    print(f"✔ {table} salva")

# ===============================================================
# SAFE COUNT
# ===============================================================

def safe_count(df):

    try:
        return df.count()
    except:
        return 0

# ===============================================================
# DATA QUALITY
# ===============================================================

def add_dq(table, bronze_df, silver_df, obs):

    dq_results.append({

        "table":table,

        "bronze":safe_count(bronze_df),

        "silver":safe_count(silver_df),

        "duplicates_removed":
            safe_count(bronze_df)-safe_count(silver_df),

        "observation":obs

    })

# ===============================================================
# LIMPEZA DE TEXTO
# ===============================================================

def clean_upper(column):

    return F.upper(F.trim(F.col(column)))

def clean_lower(column):

    return F.lower(F.trim(F.col(column)))

def clean_title(column):

    return F.initcap(F.trim(F.col(column)))

# ===============================================================
# NORMALIZA STATUS
# ===============================================================

def normalize_status(column):

    return (

        F.when(
            F.lower(F.trim(F.col(column))).isin("ativo","active"),
            F.lit("ATIVO")
        )

        .when(
            F.lower(F.trim(F.col(column))).isin("inativo","inactive"),
            F.lit("INATIVO")
        )

        .otherwise(
            F.lit("DESCONHECIDO")
        )

    )

# ===============================================================
# NORMALIZA UF
# ===============================================================

def normalize_state(column):

    return (

        F.when(
            F.upper(F.trim(F.col(column))).isin("SP","SAO PAULO","SÃO PAULO"),
            "SP"
        )

        .when(
            F.upper(F.trim(F.col(column))).isin("RJ","RIO DE JANEIRO"),
            "RJ"
        )

        .when(
            F.upper(F.trim(F.col(column))).isin("PR","PARANA"),
            "PR"
        )

        .when(
            F.upper(F.trim(F.col(column))).isin("SC","S. CATARINA","SANTA CATARINA"),
            "SC"
        )

        .when(
            F.upper(F.trim(F.col(column))).isin("MG","MINAS GERAIS"),
            "MG"
        )

        .otherwise(
            F.upper(F.trim(F.col(column)))
        )

    )

# ===============================================================
# PARSE DECIMAL
# ===============================================================

def parse_decimal(column):

    txt = F.trim(F.col(column))

    txt = F.regexp_replace(txt,",",".")

    txt = F.when(
        F.lower(txt).isin(
            "n/a",
            "null",
            "",
            "-",
            "nan"
        ),
        None
    ).otherwise(txt)

    return txt.cast(DoubleType())




# ===============================================================
# PARSE DATE (100% COMPATÍVEL SPARK CONNECT)
# ===============================================================

def parse_date(column):

    c = F.trim(F.col(column))

    # remove valores inválidos comuns
    c = F.when(
        c.isin("N/A", "NULL", "", "-", "nan"),
        None
    ).otherwise(c)

    return F.coalesce(

        # formato BR
        F.to_date(c, "dd/MM/yyyy"),

        # ISO simples
        F.to_date(c, "yyyy-MM-dd"),

        # fallback (inferência Spark)
        F.to_date(c)
    )

# ===============================================================
# PARSE TIMESTAMP (100% COMPATÍVEL SPARK CONNECT)
# ===============================================================

def parse_timestamp(column):

    c = F.trim(F.col(column))

    c = F.when(
        c.isin("N/A", "NULL", "", "-", "nan"),
        None
    ).otherwise(c)

    return F.coalesce(

        F.to_timestamp(c, "dd/MM/yyyy HH:mm"),

        F.to_timestamp(c, "yyyy-MM-dd"),

        F.to_timestamp(c, "yyyy-MM-dd'T'HH:mm:ss"),

        F.to_timestamp(c)
    )

# ===============================================================
# JSON SCHEMAS
# ===============================================================

product_schema = StructType([

    StructField("category",StringType()),

    StructField("subcategory",StringType()),

    StructField("product_id",StringType()),

    StructField("name",StringType()),

    StructField("status",StringType())

])

pricing_schema = StructType([

    StructField("currency",StringType()),

    StructField("list_price",StringType())

])

attributes_schema = StructType([

    StructField("family",StringType()),

    StructField("tags",StringType())

])

carrier_schema = StructType([

    StructField("name",StringType()),

    StructField("mode",StringType())

])

destination_schema = StructType([

    StructField("city",StringType()),

    StructField("state",StringType())

])

timestamps_schema = StructType([

    StructField("shipped_at",StringType()),

    StructField("delivered_at",StringType())

])

payment_schema = StructType([

    StructField("method",StringType()),

    StructField("installments",StringType()),

    StructField("status",StringType())

])

print("===================================================")
print("Silver Transformation V2")
print("Biblioteca carregada")
print("===================================================")

Silver Transformation V2
Biblioteca carregada


In [0]:
# ============================================================
# CLIENTES - SILVER FINAL (SPARK CONNECT SAFE)
# ============================================================

from pyspark.sql import functions as F

clientes_bronze = spark.read.format("delta").load(f"{BASE_PATH}/bronze_clientes")

clientes = (
    clientes_bronze

    # 🔥 elimina qualquer inferência do Spark
    .select([F.col(c).cast("string").alias(c) for c in clientes_bronze.columns])

    .dropDuplicates(["customer_id"])

    .withColumn("customer_id", F.col("customer_id"))

    .withColumn("nome_cliente", clean_title("nome_cliente"))

    .withColumn("email", clean_lower("email"))

    .withColumn("segmento", clean_upper("segmento"))

    .withColumn("porte", clean_upper("porte"))

    .withColumn("cidade", clean_title("cidade"))

    .withColumn("estado", normalize_state("estado"))

    .withColumn("status_cliente", normalize_status("status_cliente"))

    # ============================================================
    # DATA_CADASTRO - PARSE ROBUSTO
    # ============================================================
    .withColumn(
        "data_cadastro",
        F.when(
            F.col("data_cadastro").rlike(r"^\d{4}-\d{2}-\d{2}$"),
            F.to_date(F.col("data_cadastro"), "yyyy-MM-dd")
        )
        .when(
            F.col("data_cadastro").rlike(r"^\d{2}/\d{2}/\d{4}$"),
            F.to_date(F.col("data_cadastro"), "dd/MM/yyyy")
        )
        .otherwise(None)
    )
)

# ============================================================
# SALVAR
# ============================================================

save(clientes, "silver_clientes")

add_dq(
    "Clientes",
    clientes_bronze,
    clientes,
    "Parsing seguro de datas + padronização + deduplicação"
)

print("✔ silver_clientes criada com sucesso (FINAL FIX)")

✔ silver_clientes salva
✔ silver_clientes criada com sucesso (FINAL FIX)


In [0]:
# ============================================================
# PRODUTOS
# ============================================================

produtos_bronze = load("bronze_produtos")

produtos = (

    produtos_bronze

    .withColumn("product_json", F.from_json(F.col("product"), product_schema))

    .withColumn("pricing_json", F.from_json(F.col("pricing"), pricing_schema))

    .withColumn("attributes_json", F.from_json(F.col("attributes"), attributes_schema))

    .select(

        F.col("product_json.product_id").alias("product_id"),

        clean_title("product_json.name").alias("product_name"),

        clean_upper("product_json.category").alias("category"),

        clean_upper("product_json.subcategory").alias("subcategory"),

        clean_upper("product_json.status").alias("status"),

        clean_upper("pricing_json.currency").alias("currency"),

        parse_decimal("pricing_json.list_price").alias("list_price"),

        clean_upper("attributes_json.family").alias("family"),

        F.col("attributes_json.tags").alias("tags"),

        "updated_at",
        "_source_system",
        "_ingestion_timestamp",
        "_raw_hash"

    )

    .dropDuplicates(["product_id"])

)

save(produtos, "silver_produtos")

add_dq(
    "Produtos",
    produtos_bronze,
    produtos,
    "JSON parseado, preços normalizados e duplicados removidos"
)

print("✔ silver_produtos criada")

Lendo /Volumes/workspace/default/sources/delta/bronze_produtos
✔ silver_produtos salva
✔ silver_produtos criada


In [0]:
# ============================================================
# PEDIDOS - SILVER FINAL (ROBUSTO / SPARK CONNECT SAFE)
# ============================================================

from pyspark.sql import functions as F

pedidos_bronze = spark.read.format("delta").load(f"{BASE_PATH}/bronze_pedidos")


# ============================================================
# SAFE DATE NORMALIZER (SPARK CONNECT SAFE)
# ============================================================

def safe_date(col_name):

    c = F.trim(F.col(col_name))

    # remove lixo
    c = F.when(
        c.isin("N/A", "NULL", "", "-", "nan"),
        None
    ).otherwise(c)

    # normaliza separadores
    c = F.regexp_replace(c, "/", "-")

    # ============================================================
    # detecta formato antes de converter (evita parser quebrar)
    # ============================================================

    return F.when(
        c.rlike(r"^\d{4}-\d{2}-\d{2}$"),
        F.to_date(c, "yyyy-MM-dd")
    ).when(
        c.rlike(r"^\d{2}-\d{2}-\d{4}$"),
        F.to_date(c, "dd-MM-yyyy")
    ).otherwise(None)
# ============================================================
# TRANSFORMAÇÃO
# ============================================================

pedidos = (
    pedidos_bronze

    # 🔥 blindagem total contra schema quebrado
    .select([F.col(c).cast("string").alias(c) for c in pedidos_bronze.columns])

    .dropDuplicates(["order_id"])

    .withColumn("order_id", F.col("order_id"))

    .withColumn("customer_code", clean_upper("customer_code"))

    .withColumn("seller_id", clean_upper("seller_id"))

    .withColumn("status_order", normalize_status("status_order"))

    # ============================================================
    # DATAS SEGURAS
    # ============================================================

    .withColumn("order_date", safe_date("order_date"))

    .withColumn("promised_date", safe_date("promised_date"))
)

# ============================================================
# SALVAR
# ============================================================

save(pedidos, "silver_pedidos")

add_dq(
    "Pedidos",
    pedidos_bronze,
    pedidos,
    "Normalização completa + datas seguras + deduplicação"
)

print("✔ silver_pedidos criada com sucesso (FINAL FIX)")

✔ silver_pedidos salva
✔ silver_pedidos criada com sucesso (FINAL FIX)


In [0]:
# ============================================================
# ITENS - SILVER FINAL (ROBUSTO / SPARK SAFE)
# ============================================================

from pyspark.sql import functions as F

itens_bronze = spark.read.format("delta").load(f"{BASE_PATH}/bronze_itens")

itens = (
    itens_bronze

    # 🔥 força tudo como string (evita inferência quebrada)
    .select([F.col(c).cast("string").alias(c) for c in itens_bronze.columns])

    .dropDuplicates(["order_id", "item_seq"])

    .withColumn("order_id", F.col("order_id"))

    .withColumn("product_code", clean_upper("product_code"))

    .withColumn("item_status", normalize_status("item_status"))

    # ============================================================
    # QUANTIDADE - FIX ROBUSTO
    # ============================================================
    .withColumn(
        "quantity",
        F.when(
            F.col("quantity").rlike(r"^-?\d+\.0$"),
            F.regexp_replace(F.col("quantity"), r"\.0$", "").cast("int")
        )
        .when(
            F.col("quantity").rlike(r"^-?\d+$"),
            F.col("quantity").cast("int")
        )
        .when(
            F.col("quantity").rlike(r"^-?\d+\.?\d*"),
            F.col("quantity").cast("double").cast("int")
        )
        .otherwise(None)
    )

    # ============================================================
    # PREÇOS (DOUBLE SEGURO)
    # ============================================================
    .withColumn(
        "unit_price",
        F.regexp_replace(F.col("unit_price"), ",", ".").cast("double")
    )

    .withColumn(
        "total_item",
        F.regexp_replace(F.col("total_item"), ",", ".").cast("double")
    )
)

# ============================================================
# SALVAR
# ============================================================

save(itens, "silver_itens")

add_dq(
    "Itens",
    itens_bronze,
    itens,
    "Fix numérico robusto (int + float + limpeza de dados)"
)

print("✔ silver_itens criada com sucesso (FINAL FIX)")

✔ silver_itens salva
✔ silver_itens criada com sucesso (FINAL FIX)


In [0]:
# ============================================================
# FIX DEFINITIVO DE DATA (SEM PARSE DIRETO)
# ============================================================

# pega só números e separadores
def normalize_date(col_name):

    c = F.regexp_extract(col_name, r"(\d{4}[-/]\d{2}[-/]\d{2}|\d{2}[-/]\d{2}[-/]\d{4})", 1)

    # padroniza separador
    c = F.regexp_replace(c, "/", "-")

    # separa formatos manualmente
    yyyy_mm_dd = F.when(c.rlike(r"^\d{4}-\d{2}-\d{2}$"), c)
    dd_mm_yyyy = F.when(c.rlike(r"^\d{2}-\d{2}-\d{4}$"),
                         F.concat_ws("-",
                                     F.substring(c,7,4),
                                     F.substring(c,4,2),
                                     F.substring(c,1,2)))

    return F.coalesce(yyyy_mm_dd, dd_mm_yyyy)


# ============================================================
# USO NO DATAFRAME
# ============================================================

entregas = (
    entregas
    .withColumn("shipped_clean", normalize_date(F.col("shipped_raw")))
    .withColumn("delivered_clean", normalize_date(F.col("delivered_raw")))

    # aqui SIM agora é seguro
    .withColumn("shipped_at", F.to_date("shipped_clean"))
    .withColumn("delivered_at", F.to_date("delivered_clean"))

    .withColumn(
        "delivery_delay_days",
        F.datediff("delivered_at", "shipped_at")
    )
)

In [0]:
# ============================================================
# OCORRÊNCIAS - SILVER
# ============================================================

oc_bronze = spark.read.format("delta").load(f"{BASE_PATH}/bronze_ocorrencias")

oc = (
    oc_bronze

    .select([F.col(c).cast("string").alias(c) for c in oc_bronze.columns])

    .dropDuplicates(["ticket_id"])

    .withColumn("ticket_id", F.col("ticket_id"))

    .withColumn("customer_code", clean_upper("customer_code"))

    .withColumn("order_id", F.col("order_id"))

    .withColumn("event_type", clean_upper("event_type"))

    .withColumn("severity", clean_upper("severity"))

    .withColumn("status", normalize_status("status"))

    # ============================================================
    # METADATA (mantido simples para evitar erro de parsing JSON)
    # ============================================================
    .withColumn("metadata", F.col("metadata"))
)

save(oc, "silver_ocorrencias")

add_dq(
    "Ocorrências",
    oc_bronze,
    oc,
    "Padronização de eventos + limpeza de status"
)

print("✔ silver_ocorrencias criada")

✔ silver_ocorrencias salva
✔ silver_ocorrencias criada


In [0]:
# ============================================================
# VENDEDORES - SILVER
# ============================================================

vendedores_bronze = spark.read.format("delta").load(f"{BASE_PATH}/bronze_vendedores")

vendedores = (
    vendedores_bronze
    .select([F.col(c).cast("string").alias(c) for c in vendedores_bronze.columns])

    .dropDuplicates(["seller_id"])

    .withColumn("seller_name", clean_title("seller_name"))

    .withColumn("seller_id", F.col("seller_id"))

    .withColumn("status", normalize_status("status"))

    .withColumn("regional_code", F.upper(F.trim("regional_code")))

    .withColumn("regional_code",
        F.when(F.col("regional_code") == "S", "SUL")
        .when(F.col("regional_code") == "N", "NORTE")
        .when(F.col("regional_code") == "NE", "NORDESTE")
        .when(F.col("regional_code") == "SE", "SUDESTE")
        .when(F.col("regional_code") == "CO", "CENTRO-OESTE")
        .otherwise("DESCONHECIDO")
    )
)

save(vendedores, "silver_vendedores")
print("✔ silver_vendedores criada")

✔ silver_vendedores salva
✔ silver_vendedores criada


In [0]:
# ============================================================
# REGIÕES - SILVER
# ============================================================

regioes_bronze = spark.read.format("delta").load(f"{BASE_PATH}/bronze_regioes")

regioes = (
    regioes_bronze
    .select([F.col(c).cast("string").alias(c) for c in regioes_bronze.columns])

    .dropDuplicates(["regional_code"])

    .withColumn("regional_code", F.upper(F.trim("regional_code")))

    .withColumn("regional_name", clean_title("regional_name"))

    .withColumn("state", F.upper(F.trim("state")))

    .withColumn("manager_name", clean_title("manager_name"))

    .withColumn("active_flag", normalize_status("active_flag"))
)

save(regioes, "silver_regioes")
print("✔ silver_regioes criada")

✔ silver_regioes salva
✔ silver_regioes criada


In [0]:
# ============================================================
# CANAIS - SILVER
# ============================================================

canais_bronze = spark.read.format("delta").load(f"{BASE_PATH}/bronze_canais")

canais = (
    canais_bronze
    .select([F.col(c).cast("string").alias(c) for c in canais_bronze.columns])

    .dropDuplicates(["id_canal"])

    .withColumn("id_canal", F.upper(F.trim("id_canal")))

    .withColumn("nome_canal", clean_title("nome_canal"))

    .withColumn("tipo_canal", clean_upper("tipo_canal"))

    .withColumn("ativo", normalize_status("ativo"))

    .withColumn("observacao", F.col("observacao"))
)

save(canais, "silver_canais")
print("✔ silver_canais criada")

✔ silver_canais salva
✔ silver_canais criada
